# Train MiniGPT on TinyStories (byte-level BPE)

Reproducible training notebook for the **MiniGPT** project: a from-scratch GPT-style
language model in PyTorch. It trains the byte-level BPE + TinyStories configuration on
a single GPU (e.g. a free Colab T4) and saves the artifacts the API serves.

Steps:
1. Setup (clone the repo + install training deps)
2. Configure the model and training run
3. Train (downloads TinyStories, trains the BPE tokenizer, trains the model)
4. Generate samples from the trained model
5. Download the artifacts and serve them

> All modeling code lives in `src/` and is imported here - the notebook only wires
> together `train()` and `generate`, so what you train is exactly what the API serves.

To run on Colab: open this notebook, set `Runtime > Change runtime type > GPU`, then set
`REPO_URL` below to your fork and run all cells.

## 1. Setup

Clone the repository and install the training-only dependencies. Colab already ships
`torch` and `numpy` with CUDA, so we install just the extras and keep Colab's GPU build
of PyTorch (we do **not** install `requirements.txt`, which pins the CPU serving stack).

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Ilyes-Jamoussi/minigpt-llm.git"

# Clone only when running outside the repo (e.g. a fresh Colab runtime).
if not Path("src").exists():
    !git clone -q $REPO_URL repo
    os.chdir("repo")

!pip install -q safetensors datasets tqdm
print("working directory:", Path.cwd())

## 2. Configure

All knobs live in `TrainConfig` / `GPTConfig` (the project's single source of truth).
Defaults below target a ~10M-parameter model and a few hours on one T4. Tune
`max_chars`, `max_steps`, and `batch_size` to your time and memory budget. `vocab_size`
is corrected from the trained tokenizer automatically, so the placeholder is fine.

In [ ]:
import logging

from src.config import GPTConfig, TrainConfig, resolve_device

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
print("device:", resolve_device())

train_config = TrainConfig(
    dataset="tinystories",
    tokenizer_type="bpe",
    bpe_vocab_size=8192,
    max_chars=100_000_000,  # ~100 MB of TinyStories; lower this to train faster
    batch_size=32,
    max_steps=20_000,
    learning_rate=3e-4,
    min_lr=3e-5,
    warmup_steps=200,
    weight_decay=0.1,
    grad_clip=1.0,
    eval_interval=500,
    eval_iters=100,
    sample_interval=2_000,
    sample_prompt="Once upon a time",
    sample_max_new_tokens=200,
    seed=1337,
)

model_config = GPTConfig(
    vocab_size=train_config.bpe_vocab_size,  # overwritten from the tokenizer in train()
    block_size=256,
    n_layer=6,
    n_head=6,
    d_model=384,
    dropout=0.1,
)

## 3. Train

`train()` downloads TinyStories, trains the byte-level BPE tokenizer on the train split,
trains the model with a cosine schedule and mixed precision (on CUDA), logs validation
loss/perplexity, prints periodic samples, and checkpoints the best model to `models/`
(`model.safetensors`, `config.json`, `tokenizer.json`, `metrics.json`).

> The first run spends a few minutes training the BPE tokenizer (pure Python). Lower
> `bpe_vocab_size` if you want it faster.

In [ ]:
from src.train import train

metrics = train(train_config, model_config)
metrics

## 4. Generate samples

Reload the best checkpoint exactly as the API does (`load_pretrained`) and sample a few
completions.

In [ ]:
import torch

from src.config import MODELS_DIR, resolve_device
from src.model import load_pretrained

device = resolve_device()
model, tokenizer = load_pretrained(MODELS_DIR, device)

for prompt in ["Once upon a time", "The little robot", "Lily and Tom went to"]:
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=50, top_p=0.95)
    print(tokenizer.decode(out[0].tolist()))
    print("-" * 60)

## 5. Download the artifacts and serve

The trained artifacts are in `models/`. The reported **validation perplexity** and
token budget are in `models/metrics.json` (read it and copy the numbers into the
README's results section).

To serve them:
1. Download the four files in `models/` (`model.safetensors`, `config.json`,
   `tokenizer.json`, `metrics.json`) and place them in your local repo's `models/`.
2. Run the API locally: `uvicorn api.main:app` and open `http://localhost:8000/`.
3. Or build the Docker image and deploy (see the README's deployment note).

In [ ]:
import json

from src.config import METRICS_FILENAME, MODELS_DIR

print(json.dumps(json.loads((MODELS_DIR / METRICS_FILENAME).read_text()), indent=2))